# Session 03 — Draw a helpful camera guide

**Today’s new skill:** use drawings on an image or webcam frame to tell a person what to do.

The system looks for an orange object, then draws a target and gives one clear response: **READY**, **ADJUST**, or **LOST**.

## Choose your route

- **In Colab:** upload one photo containing an orange object. The notebook will draw the guide on that photo.
- **On your own computer:** run the local webcam cell for a live version.

Colab cannot open `cv2.VideoCapture(0)`, so it uses an uploaded image instead of a live webcam loop.

In [ ]:
# Run this once if OpenCV is not installed.
%pip install opencv-python

# OpenCV reads images and draws the visual guide.
import cv2

## 1. Choose the rule you will test

Start with the suggested value. Later, change **only** `READY_TOLERANCE` and observe how the decision changes.

In [ ]:
# Orange colour range. Leave these two lines alone for the first test.
ORANGE_LOW = (4, 120, 90)
ORANGE_HIGH = (24, 255, 255)

# Size of the blue target box in the middle of the frame.
TARGET_WIDTH = 360
TARGET_HEIGHT = 240

# ★ YOUR FIRST EDIT: how close must the object be to the target centre?
# Try 24 first. Then try 50. Which value gives the most useful feedback?
READY_TOLERANCE = 24

## 2. Find the centre of the orange object

This helper returns the middle of the largest orange shape. If it cannot see an orange object, it returns `None`.

In [ ]:
def find_orange_centre(frame):
    # HSV makes it easier to keep only orange pixels.
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    orange_mask = cv2.inRange(hsv, ORANGE_LOW, ORANGE_HIGH)

    # Find orange shapes and keep the largest one.
    contours, _ = cv2.findContours(
        orange_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    if not contours:
        return None

    biggest_shape = max(contours, key=cv2.contourArea)
    if cv2.contourArea(biggest_shape) < 700:
        return None

    # Calculate the centre point of that shape.
    moments = cv2.moments(biggest_shape)
    if moments['m00'] == 0:
        return None
    centre_x = int(moments['m10'] / moments['m00'])
    centre_y = int(moments['m01'] / moments['m00'])
    return (centre_x, centre_y)

## 3. Draw the guide and choose a status

This helper adds the blue target, the centre cross, the measured point, and the message shown to the user.

In [ ]:
def draw_alignment_guide(frame):
    # Make a copy so the original frame stays unchanged.
    output = frame.copy()
    frame_height, frame_width = output.shape[:2]

    # Put the target box in the middle of this frame.
    left = (frame_width - TARGET_WIDTH) // 2
    top = (frame_height - TARGET_HEIGHT) // 2
    right = left + TARGET_WIDTH
    bottom = top + TARGET_HEIGHT
    target_centre = (left + TARGET_WIDTH // 2, top + TARGET_HEIGHT // 2)

    blue = (21, 138, 194)
    orange = (246, 138, 69)

    # Draw the visible guide for the user.
    cv2.rectangle(output, (left, top), (right, bottom), blue, 3)
    cv2.drawMarker(output, target_centre, orange, cv2.MARKER_CROSS, 20, 2)

    # Start with LOST. This changes only if we find an orange object.
    object_centre = find_orange_centre(output)
    status = 'LOST — show orange object'
    status_colour = orange

    if object_centre is not None:
        object_x, object_y = object_centre

        # Measure how far the orange object is from the target centre.
        distance = ((object_x - target_centre[0]) ** 2 +
                    (object_y - target_centre[1]) ** 2) ** 0.5
        inside_box = left <= object_x <= right and top <= object_y <= bottom
        ready = inside_box and distance <= READY_TOLERANCE

        # Draw the point that the system measured.
        status_colour = blue if ready else orange
        cv2.circle(output, object_centre, 8, status_colour, -1)
        cv2.line(output, target_centre, object_centre, status_colour, 2)

        # Choose the next action for the user.
        status = 'READY — capture frame' if ready else 'ADJUST — centre orange object'

    # Put the decision and the editable rule on the frame.
    cv2.putText(output, status, (18, 35), cv2.FONT_HERSHEY_SIMPLEX, .65, status_colour, 2)
    cv2.putText(output, f'tolerance: {READY_TOLERANCE}px', (18, frame_height - 18),
                cv2.FONT_HERSHEY_SIMPLEX, .6, (255, 255, 255), 2)
    return output, status

## 4A. Colab route — upload one photo

Use a photo with one orange object. This works in Colab and lets you test the same visible guide without a webcam.

In [ ]:
# Colab opens a file picker. Choose a photo with an orange object.
try:
    from google.colab import files
    from google.colab.patches import cv2_imshow
except ImportError:
    raise RuntimeError('This cell is for Colab. Use the local webcam cell below instead.')

uploaded = files.upload()
photo_name = next(iter(uploaded))
photo = cv2.imread(photo_name)

# Draw the guide on the uploaded photo and show the decision.
result, status = draw_alignment_guide(photo)
cv2_imshow(result)
print(status)

## 4B. Local route — run the live webcam

Run this only on your own computer, in a local Python or Jupyter environment. Press **q** in the camera window to stop.

In [ ]:
# Open the first webcam on this computer.
camera = cv2.VideoCapture(0)

if not camera.isOpened():
    print('No local webcam was found. In Colab, use the upload-photo cell above.')
else:
    while True:
        # Read one new frame, draw the guide, and show it.
        ok, frame = camera.read()
        if not ok:
            break
        result, status = draw_alignment_guide(frame)
        cv2.imshow('Alignment guide — press q to exit', result)

        # Press q in the camera window when you are finished.
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Close the camera safely after the loop ends.
    camera.release()
    cv2.destroyAllWindows()

## 5. Record evidence

1. With the object **centred**, what status appears?
2. With it **off-centre**, what status appears?
3. When it is **out of view**, what status appears?
4. After changing `READY_TOLERANCE`, does the guide approve too early, too late, or at a useful point?

**Exit evidence:** write one rule you changed, one visible result, and one case where a user could still be confused.